# Model Trainer Stage

This notebook mirrors the training stage: load the prepared model, build train and validation generators, and fit the classifier.

In [ ]:
from pathlib import Path
import yaml
import tensorflow as tf

with open("params.yaml", "r", encoding="utf-8") as file_obj:
    params = yaml.safe_load(file_obj)

TRAINING_DATA = Path("artifacts/data_ingestion/Chest-CT-Scan-data")
MODEL_PATH = Path("artifacts/prepare_base_model/base_model_updated.h5")

params, TRAINING_DATA, MODEL_PATH

In [ ]:
datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)
dataflow_kwargs = dict(
    target_size=tuple(params["IMAGE_SIZE"][:-1]),
    batch_size=params["BATCH_SIZE"],
    interpolation="bilinear",
    class_mode="categorical"
)

valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)

if params["AUGMENTATION"]:
    train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        **datagenerator_kwargs
    )
else:
    train_datagenerator = valid_datagenerator

In [ ]:
train_generator = train_datagenerator.flow_from_directory(
    directory=str(TRAINING_DATA),
    subset="training",
    shuffle=True,
    **dataflow_kwargs
)

valid_generator = valid_datagenerator.flow_from_directory(
    directory=str(TRAINING_DATA),
    subset="validation",
    shuffle=False,
    **dataflow_kwargs
)

train_generator.class_indices

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=params["EPOCHS"]
)

output_dir = Path("artifacts/training")
output_dir.mkdir(parents=True, exist_ok=True)
model.save(output_dir / "model.h5")
print("Trained model saved")